# 01 Fetch ECMWF open data: AIFS-single and IFS

Downloads the forecast runs in the configured window from data.ecmwf.int, caches the GRIB2 files with a manifest, and writes one standardised NetCDF field file per (model, initialisation) cut to the Norwegian bbox.

## What this notebook does and does not show

Does: `latest_available`, the run loop over `runs_in_window` for both models with skip reasons printed, the equivalent CLI call, one `pcolormesh` of 2 m temperature at lead 24 h, and the manifest of one cached GRIB.

Does not: touch observations, match anything to stations, or produce a score. The map is a sanity check on the domain cut and the unit conversion, not a result.

## Facts about the source (checked 7 Sep 2026)

* Files sit on data.ecmwf.int for about four days, so anything older than that is gone and the loop below prints a skip for it. Cache daily.
* Both `model="aifs-single"` (data-driven) and `model="ifs"` (physics HRES) are served at 0.25 degree only and start from the same initial conditions, which is what makes the pairwise comparison fair.
* Steps are 6 hourly. The config uses 6 to 72 h.
* `tp` is accumulated from the forecast start, in metres in the GRIB; `standardise` converts to mm and `afv.match.deaccumulate` turns it into window totals.
* A run appears 7 to 9 h after its base time. The 00 UTC run is usable from about 09 UTC, the 12 UTC run from about 21 UTC.
* IFS uses stream `oper` for 00 and 12 UTC and `scda` for 06 and 18 UTC; `fetch_run` sets this.

In [ ]:
import os
from pathlib import Path

from afv.config import load_config

CONFIG = Path(os.environ.get("AFV_CONFIG", "configs/default.yaml"))
cfg = load_config(CONFIG)
runs = cfg["runs"]
print(CONFIG, "->", cfg["name"])
print("models:", cfg["models"])
print("window:", runs["window_start"], "to", runs["window_end"], "| init hours", runs["init_hours"])
print("leads:", runs["lead_hours"])
print("bbox:", cfg["stations"]["bbox"])

The next cell uses the network: one small request to data.ecmwf.int for the newest available AIFS-single run.

In [ ]:
from afv.fetch_forecast import fetch_and_standardise, fetch_run, latest_available, runs_in_window

latest = latest_available("aifs-single")
print("latest AIFS-single run on data.ecmwf.int:", latest)

The next cell uses the network and downloads several hundred MB the first time. Every run in the window is tried for both models; runs in the future or already expired are reported as skips and the loop keeps going. A second call is served from the cache.

In [ ]:
from datetime import datetime, timezone

steps = list(runs["lead_hours"])
inits = runs_in_window(runs["window_start"], runs["window_end"], list(runs["init_hours"]))
now = datetime.now(timezone.utc)
produced = []
for init in inits:
    if init > now:
        print(f"skip {init:%Y-%m-%dT%HZ}: in the future")
        continue
    for model in cfg["models"]:
        try:
            p = fetch_and_standardise(model, init, steps, cfg.get_path("cache_dir"),
                                      cfg.get_path("fields_dir"), cfg["stations"].get("bbox"))
            produced.append(p)
            print(f"ok   {model:12s} {init:%Y-%m-%dT%HZ} -> {p.name}")
        except Exception as e:
            print(f"skip {model:12s} {init:%Y-%m-%dT%HZ}: {e}")
print(len(produced), "field files written or already present")

`fetch_run` alone downloads without decoding, which is what `afv fetch --download-only` does. Use it on a machine without eccodes and run the standardise step where a decoder exists.

## The same thing from the command line

The next cell uses the network. It is the CLI equivalent of the loop above; `--latest` fetches only the newest run and `--model aifs-single` restricts to one model.

In [ ]:
# Network. Same as: afv fetch --config configs/default.yaml
!afv fetch --config {CONFIG}

## Look at one field file

No network from here on. Field files have dims `(lead_h, lat, lon)`, SI units, and the attributes `model` and `init_time`.

In [ ]:
import xarray as xr

fields = sorted(cfg.get_path("fields_dir").glob("*.nc"))
print(len(fields), "field files")
for f in fields[:6]:
    print(" ", f.name)
ds = xr.open_dataset(fields[0])
ds

In [ ]:
import matplotlib.pyplot as plt

t = ds["t2m"].sel(lead_h=24) - 273.15
fig, ax = plt.subplots(figsize=(7, 5))
pm = ax.pcolormesh(ds["lon"], ds["lat"], t, shading="auto", cmap="RdBu_r")
fig.colorbar(pm, ax=ax, label="2 m temperature (deg C)")
ax.set_title(f"{ds.attrs['model']} init {ds.attrs['init_time']} lead 24 h, 0.25 degree")
ax.set_xlabel("lon")
ax.set_ylabel("lat")
plt.show()

## Provenance of one cached GRIB

`fetch_run` writes a `.manifest.json` next to each GRIB with the sha256, the byte count, the exact request and the fetch time. This is what makes a later re-run checkable.

In [ ]:
import json

manifests = sorted((cfg.get_path("cache_dir") / "grib").glob("*.manifest.json"))
print(len(manifests), "manifests")
if manifests:
    print(json.dumps(json.loads(manifests[0].read_text(encoding="utf-8")), indent=2))